# 03b Preliminary table metrics

Values for the preliminary results table that need no student training.

- **Params, serialized size, and CPU latency** depend only on the architecture, not on trained weights, so they are measured for the teacher and the student (both students share one architecture).
- **Historical control** metrics need no training at all.
- **Teacher** metrics use the three seed checkpoints committed by notebook 03.

Nothing is written to disk, so this notebook does not mount Google Drive.

## Setup

In [1]:
# setup: clone repo (or pull latest), install deps; read-only notebook, so no drive mount
import os, sys, subprocess

REPO_URL = "https://github.com/BerkayPolat-lab/gru-currency-volatility-distillation.git"
REPO_DIR = "/content/gru-currency-volatility-distillation"

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "-q", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "-q"], check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{REPO_DIR}/model/requirements.txt"], check=True)
os.chdir(f"{REPO_DIR}/model")  # all paths in config relative to model/
sys.path.insert(0, f"{REPO_DIR}/model/src")  # lets notebooks do: from gru_vol import ...
print("ready, working dir:", os.getcwd())

ready, working dir: /content/gru-currency-volatility-distillation/model


In [2]:
import yaml

with open("configs/base.yaml") as f:
    CFG = yaml.safe_load(f)

## Params and serialized size

Size is the saved state dict (weights only, no optimizer state) in KB (1 KB = 1024 bytes).

In [3]:
# params + serialized size depend only on the architecture, not on trained weights
import json
import numpy as np
import pandas as pd
import torch
from gru_vol.models import build_model
from gru_vol.benchmark import param_count, serialized_size_kb

arch = {"teacher": build_model(CFG, "teacher"), "student": build_model(CFG, "student")}
size_table = pd.DataFrame.from_dict({kind: {"params": param_count(m), "size_kb": serialized_size_kb(m)} for kind, m in arch.items()}, orient="index")
size_table.round(1)

,params,size_kb
teacher,249729,979.9
student,3681,17.0


## CPU latency

Batch-1 forward pass over one full 60-step window, on the CPU with the thread count, warm-up runs, and trials from `base.yaml`.
Every model is timed in the same session, so the numbers are comparable; Colab CPUs vary between sessions, so copy the printed CPU name into the table caption.

In [6]:
!lscpu

Architecture:                x86_64
  CPU op-mode(s):            32-bit, 64-bit
  Address sizes:             46 bits physical, 48 bits virtual
  Byte Order:                Little Endian
CPU(s):                      2
  On-line CPU(s) list:       0,1
Vendor ID:                   GenuineIntel
  Model name:                Intel(R) Xeon(R) CPU @ 2.20GHz
    CPU family:              6
    Model:                   79
    Thread(s) per core:      2
    Core(s) per socket:      1
    Socket(s):               1
    Stepping:                0
    BogoMIPS:                4400.41
    Flags:                   fpu vme de pse tsc msr pae mce cx8 apic sep mtrr pg
                             e mca cmov pat pse36 clflush mmx fxsr sse sse2 ss h
                             t syscall nx pdpe1gb rdtscp lm constant_tsc rep_goo
                             d nopl xtopology nonstop_tsc cpuid tsc_known_freq p
                             ni pclmulqdq ssse3 fma cx16 pcid sse4_1 sse4_2 x2ap
                   

In [9]:
# cpu latency: batch 1, one full 60 step window, same cpu + thread count for every model
from gru_vol.benchmark import cpu_latency_ms
from gru_vol.data import load_prices, log_returns

npz = np.load("data/processed/splits.npz")
x1 = torch.as_tensor(npz["X_validation"][:1])  # (1, 60, 4) scaled window
hc_window = (log_returns(load_prices(CFG)) ** 2).iloc[-CFG["historical_control"]["window"]:].to_numpy()  # last 60 squared returns

latency = {}
with torch.no_grad():
    for kind, m in arch.items():
        latency[kind] = cpu_latency_ms(m.eval(), x1, CFG)  # speed does not depend on weight values
latency["hc"] = cpu_latency_ms(np.mean, hc_window, CFG)  # hc forecast = mean of the window

cpu_name_hardcoded = "Intel(R) Xeon(R) CPU @ 2.20GHz"

b = CFG["benchmark"]
print(f"cpu: {cpu_name_hardcoded} | threads: {b['num_threads']} | {b['timed_runs']} trials after {b['warmup_runs']} warm-up runs")
pd.DataFrame(latency, index=["p50_ms", "p95_ms"]).T.round(3)

cpu: Intel(R) Xeon(R) CPU @ 2.20GHz | threads: 1 | 1000 trials after 50 warm-up runs


,p50_ms,p95_ms
teacher,5.406,7.058
student,1.478,1.711
hc,0.005,0.005


## Historical control (validation)

The control forecast is the trailing 60-observation variance (a ratio of 1), so it needs no training.
MSE is shown in 1e-10 units; the high-volatility subset covers only the validation dates above the training cutoff, so treat it as descriptive.

In [10]:
# historical control needs no training: forecast = trailing 60 day variance
from gru_vol.evaluation import mse, high_vol_mse

v60, tgt = npz["var60_validation"], npz["target_validation"]
cutoff = json.load(open("artifacts/manifests/data_manifest.json"))["target"]["high_vol_cutoff"]
hc_val = mse(tgt, v60)
hc_hv, n_hv = high_vol_mse(tgt, v60, cutoff)
print(f"hc | val mse {hc_val * 1e10:.3f} | high-vol mse {hc_hv * 1e10:.1f} (n = {n_hv}) | r2 {np.corrcoef(v60, tgt)[0, 1] ** 2:.3f}")

hc | val mse 2.571 | high-vol mse 102.1 (n = 8) | r2 0.068


## Teacher (validation)

Each committed seed checkpoint is scored in original units (predicted ratio times trailing variance), then summarized as mean ± standard deviation over seeds.

In [11]:
# teacher row: reload each seed checkpoint from notebook 03 and score it in original units
from gru_vol.training import predict

device = "cuda" if torch.cuda.is_available() else "cpu"
rows = []
for s in CFG["training"]["seeds"]:
    m = build_model(CFG, "teacher").to(device)
    m.load_state_dict(torch.load(f"artifacts/checkpoints/teacher_seed{s}.pt", map_location=device))
    f = predict(m, npz["X_validation"], device) * v60  # ratio * trailing variance
    rows.append({"seed": s, "val_mse": mse(tgt, f) * 1e10, "vs_hc_pct": 100 * (1 - mse(tgt, f) / hc_val),
                 "high_vol_mse": high_vol_mse(tgt, f, cutoff)[0] * 1e10, "r2": np.corrcoef(f, tgt)[0, 1] ** 2})

teacher_row = pd.DataFrame(rows).set_index("seed")
print(teacher_row.round(3))
teacher_row.agg(["mean", "std"]).round(3)

      val_mse  vs_hc_pct  high_vol_mse     r2
seed                                         
0       2.438      5.183        96.639  0.098
1       2.453      4.577        94.499  0.099
2       2.460      4.295        93.786  0.101


,val_mse,vs_hc_pct,high_vol_mse,r2
mean,2.450,4.685,94.975,0.099
std,0.012,0.454,1.485,0.001
